# Stackformer Quickstart: GPT-2 Model Training

This tutorial walks through building and training a GPT-2 language model using Stackformer's modular components and high-level `Trainer`.

In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset

from stackformer.config import ModelConfig
from stackformer.models import GPT2
from stackformer.training import Trainer, TrainConfig
from stackformer.utils.checkpoint import load_model_weights

## 1. Configure and Instantiate GPT-2

In [ ]:
config = ModelConfig(
    vocab_size=1000,
    embed_dim=128,
    num_layers=4,
    num_heads=4,
    seq_len=64,
    hidden_dim=256,
    dropout=0.1,
)

model = GPT2(config)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## 2. Prepare Training and Validation Data

In [ ]:
torch.manual_seed(42)
train_x = torch.randint(0, 1000, (64, 64))
train_y = torch.randint(0, 1000, (64, 64))
train_loader = DataLoader(TensorDataset(train_x, train_y), batch_size=8, shuffle=True)

val_x = torch.randint(0, 1000, (16, 64))
val_y = torch.randint(0, 1000, (16, 64))
val_loader = DataLoader(TensorDataset(val_x, val_y), batch_size=8)

## 3. Set Up Trainer & Train

In [ ]:
train_cfg = TrainConfig(
    max_epochs=2,
    lr=1e-3,
    checkpoint_dir="./checkpoints_quickstart",
    eval_every_n_epochs=1,
    display="auto",
    verbosity="normal",
)

trainer = Trainer(model, train_cfg, train_loader=train_loader, eval_loader=val_loader)
state = trainer.train()
print(f"Training finished! Total steps: {state.global_step}")

## 4. Export & Reload Weights via Safetensors

In [ ]:
trainer.save("./checkpoints_quickstart/final_model.safetensors")

new_model = GPT2(config)
load_model_weights(new_model, "./checkpoints_quickstart/final_model.safetensors")
print("Weights reloaded successfully into fresh model!")